# 00: Data Dictionary & Metadata Mapping
***
**Author:** Zain Zabidi
**Date:** February 2026

**Objective:** To transform categorical ABS Census bins (Rent, Income, Mortgage)

## 1. Environment Setup & Data Loading
We load specific ABS datasets thhat contained **binned categorical ranges** (e.g., "$150-$199). These require numerical mapping before they can be used for analysis.

**Datasets requiring midpoint dictionaries:**
* **G40 (Target):** Weekly Rent Data.
* **G33 (Predictor):** Total Household Income.
* **G38 (Predictor):** Monthly Mortgage Repayments.

*Note: Other predictors, such as Industry (G54) and Labour Force (G46) data, are already provided as counts and will be processed in later notebooks.*

### 1.1 Import Libraries and Load Data
We use **Pandas** for data manipulation and **Regualar Expressions (re)** to help us parse numbers out of the column strings.

In [8]:
import pandas as pd
import os
import re

# Define paths for easier referencing
RAW_DATA_PATH = '../data/raw/'
DICT_PATH = '../data/dictionary/'

# Ensure dict dir exists
os.makedirs(DICT_PATH, exist_ok=True)

# Load core datasets
df_g40 = pd.read_csv(os.path.join(RAW_DATA_PATH, '2021Census_G40_WA_SAL.csv'))
df_g33 = pd.read_csv(os.path.join(RAW_DATA_PATH, '2021Census_G33_WA_SAL.csv'))
df_g38 = pd.read_csv(os.path.join(RAW_DATA_PATH, '2021Census_G38_WA_SAL.csv'))

print(f"G40 (Rent) loaded: {df_g40.shape}")
print(f"G33 (Income) loaded: {df_g33.shape}")
print(f"G38 (Mortgage) loaded: {df_g38.shape}")

G40 (Rent) loaded: (1701, 113)
G33 (Income) loaded: (1701, 61)
G38 (Mortgage) loaded: (1701, 79)


### 1.2 Isolating Total Brackets
The raw ABS datasets contain numerous sub-category columns (e.g., rent broken down by landlord type or income by household composition). 

For this analysis, we isolate the **Total** counts for each bracket, identified by the `_Tot` suffix. We extract these headers for Rent (G40), Income (G33), and Mortgage (G38) to prepare them for numerical conversion.

In [9]:
# Extract 'Total' columns for each dataset
rent_cols = [c for c in df_g40.columns if c.endswith('_Tot') and c != 'Tot_Tot']
inc_cols  = [c for c in df_g33.columns if c.endswith('_Tot') and c != 'Tot_Tot']
mort_cols = [c for c in df_g38.columns if c.endswith('_Tot') and c != 'Tot_Tot']

print(f"Identified {len(rent_cols)} Rent brackets (G40)")
print(f"Identified {len(inc_cols)} Income brackets (G33)")
print(f"Identified {len(mort_cols)} Mortgage brackets (G38)")

Identified 15 Rent brackets (G40)
Identified 19 Income brackets (G33)
Identified 11 Mortgage brackets (G38)


## 2. Midpoint Calculation Logic
To convert categorical brackets into a continuous numberical variable, we calculate the midpoint for each range.

For example, a bracket labeled **$150-199** is converted to **$174.50**. We use Regualr Expressions (Regex) to find the numeric values within the column strings and apply a mathematical average.

**Special Casses:**
- **"Over" brackets:** For the top-end bracket (e.g., $950 and over), we apply a 20% buffer too the base value to estimate a representative midpoint.
- **"Not stated" or "Nil":** These are handled as zero or null depending on the context.

In [10]:
def calculate_midpoint(column_name, top_bracket_multiplier=1.2):
    """
    Parses an ABS column header to find numbers and return a midpoint.
    Example: 'R_150_199_Tot' -> 174.5
    """
    # Extracting numbers from string
    numbers = [int(n) for n in re.findall(r'\d+', column_name)]

    # Logic for standard ranges (e.g., [150, 199])
    if len(numbers) == 2:
        return sum(numbers) / 2
    
    # Logic for 'over' brackets (e.g., [950])
    elif len(numbers) == 1:
        return numbers[0] * top_bracket_multiplier
    
    # For 'Nil' or 'Not Stated'
    return 0

# Test the function
test_headers = ['R_150_199_Tot', 'R_950_over_Tot']
for h in test_headers:
    print(f"Header: {h} -> Estimated Midpoint: ${calculate_midpoint(h):.2f}")

Header: R_150_199_Tot -> Estimated Midpoint: $174.50
Header: R_950_over_Tot -> Estimated Midpoint: $1140.00


## 2.1 Generating Mapping Dictionaries
With the `Calculate_Midpoint` function validated, we now apply it to the complete lists of Rent, Income, and Mortgage brackets.

This process generates a key-value mapping where each Census column header is paired with its numerical midpoint. These dictionaries will eventually be exported as JSON files for use in the data cleaning and analysis notebooks.

In [11]:
# Create dicts using dict comprehension
rent_map = {col: calculate_midpoint(col) for col in rent_cols}
inc_map = {col: calculate_midpoint(col) for col in inc_cols}
mort_map = {col: calculate_midpoint(col) for col in mort_cols}

# Preview mappings
print("Rent Mapping (Sample):")
print(dict(list(rent_map.items())[:3]))

print("\nIncome Mapping (Sample):")
print(dict(list(inc_map.items())[:3]))

print("\nMortgage Mapping (Sample):")
print(dict(list(mort_map.items())[:3]))

Rent Mapping (Sample):
{'R_1_74_Tot': 37.5, 'R_75_99_Tot': 87.0, 'R_100_149_Tot': 124.5}

Income Mapping (Sample):
{'Negative_Nil_income_Tot': 0, 'HI_1_149_Tot': 75.0, 'HI_150_299_Tot': 224.5}

Mortgage Mapping (Sample):
{'M_0_299_Tot': 149.5, 'M_300_449_Tot': 374.5, 'M_450_599_Tot': 524.5}


## 3. Exporting Mappings to JSON
To ensure consistency across the project, we export these dictionaries as JSON files. This allows other notebooks to load the midpoint values without re-running the parsing logic.

In [12]:
import json

# Ensure dict dir exists
os.makedirs(DICT_PATH, exist_ok=True)

# Save the mappings
mappings = {
    'rent_midpoints.json': rent_map,
    'income_midpoints.json': inc_map,
    'mortgage_midpoints.json': mort_map
}

for filename, data in mappings.items():
    file_full_path = os.path.join(DICT_PATH, filename)
    with open(file_full_path, 'w') as f:
        json.dump(data, f, indent=4)

print(f"Succefully exported {len(mappings)} dictionaries to: {DICT_PATH}")

Succefully exported 3 dictionaries to: ../data/dictionary/


## Summary & Next Steps
In this notebook, we converted ABS categorical rent, income, and mortgage brackets into numerical midpoints.

**Outputs:**
- `rent_midpoints.json`
- `income_midpoints.json`
- `mortgage_midpoints.json`

These dictionaries are saved in `data/dictionary/` and will be used in the next notebook (**01_data_cleaning.ipynb**) to transform raw census counts into meaningful averages for each **suburb** across Western Australia.

***Note on Non-Numerical Categories:**
Brackets such as "Not Stated" or "Partial Income" have been assigned a midpoint of `0`. These will be treated as missing data and excluded from weighted average calculations in the cleaning phase to ensure suburb averages remain accurate.*